In [119]:
import random
from collections import deque

In [120]:
class Tile:
    
    def __init__(self, tuple_tile: tuple[int, int]):
        self.left_side = tuple_tile[0]
        self.right_side = tuple_tile[1]


    def __str__(self) -> str:
        return f"[{self.left_side}|{self.right_side}]"


    def __repr__(self) -> str:
        return self.__str__()


    def __eq__(self, other: Tile) -> bool:
        if not isinstance(other, Tile):
            return NotImplemented
        return (
            (self.left_side == other.left_side 
             and self.right_side == other.right_side)
            or 
            (self.left_side == other.right_side 
             and self.right_side == other.left_side)
        )


    def __iter__(self):
        yield self.left_side
        yield self.right_side


    def __add__(self, other):
        if isinstance(other, Tile):
            return (
                self.left_side + self.right_side
                + other.left_side + other.right_side
            )
        return NotImplemented


    @staticmethod
    def swap_tile(tile: Tile) -> Tile:
        tile.left_side, tile.right_side = tile.right_side, tile.left_side
        return tile


    def is_double(self) -> bool:
        return True if self.left_side ==  self.right_side else False

In [121]:
class Table:

    def __init__(self):
        self.tiles_deque = deque()
        self.left_side = None
        self.right_side = None


    def __str__(self) -> str:
        if not self.tiles_deque:
            return "Mesa vazia"

        return "".join(str(tile) for tile in self.tiles_deque)


    def __contains__(self, tile: Tile):
        return tile in self.tiles_deque


    def __getitem__(self, index: int) -> Tile:
        return self.tiles_deque[index]


    def get_sides(self):
        return [self.left_side, self.right_side]


    def add_first_tile(self, initial_tile: Tile):
        self.left_side = initial_tile.left_side
        self.right_side = initial_tile.right_side

        self.tiles_deque.append(initial_tile)


    def add_left(self, tile: Tile) -> None:
        if tile.right_side != self.left_side:
            if tile.left_side != self.left_side:
                raise ValueError("Não vale roubar!!")

            Tile.swap_tile(tile)

        self.tiles_deque.appendleft(tile)
        self.left_side = tile.left_side


    def add_right(self, tile: Tile) -> None:
        if tile.left_side != self.right_side:
            if tile.right_side != self.right_side:
                raise ValueError("Não vale roubar!!")

            Tile.swap_tile(tile)

        self.tiles_deque.append(tile)
        self.right_side = tile.right_side


    def show_table(self):
        if not self.tiles_deque:
            print(self)
        else:
            print("\n"+99*"-")
            print(self)
            print(99*"-"+"\n")


In [122]:
class Player:

    def __init__(self, name: str, tiles: list[Tile]):
        self.name = name
        self.tiles = tiles
        self.wins = 0


    def __str__(self):
        return f"{self.name}\n{self.tiles}"


    def __repr__(self) -> str:
        return f"{self.name}\n{self.tiles}"


    def get_tile_points(self) -> int:
        return sum(tile.left_side + tile.right_side for tile in self.tiles)


    def can_play(self, table: Table) -> bool:

        if not self.tiles:
            return False

        if not table.tiles_deque:
            return True

        for t in self.tiles:
            if (
                t.left_side == table.left_side or t.left_side == table.right_side 
                    or 
                t.right_side == table.right_side or t.right_side == table.left_side
            ):
                return True
            
        return False


    def get_playable_tiles(self, table: Table, verbose: bool = False, has_sleep: bool = False):

        if not table.tiles_deque:
            playable_tiles = self.tiles.copy()
        else:
            playable_tiles = [
                t for t in self.tiles 
                if (
                    t.left_side in (
                        table.left_side,
                        table.right_side
                    )   
                    or t.right_side in(
                        table.left_side,
                        table.right_side
                    )
                )
            ]

        if verbose:
            width = 8
            has_playable_tiles = bool(playable_tiles)

            pieces_row = "".join(
                str(tile).center(width) for tile in self.tiles
            )

            indices = []
            playable_idx = 0

            for tile in self.tiles:
                if tile in playable_tiles:
                    indices.append(str(playable_idx).center(width))
                    playable_idx += 1
                else:
                    indices.append(" " * width)

            indices_row = "".join(indices)

            if not has_playable_tiles:
                if has_sleep:
                    sleep_art = [
                        "  _______ ",
                        " /______/|",
                        "| SLEEP ||",
                        "|_______|/",
                        " [ENTER] "
                        "\n"
                    ]
                    
                    print(pieces_row + "   " + sleep_art[0])
                    print(indices_row + "   " + sleep_art[1])
                    print(" " * len(indices_row) + "   " + sleep_art[2])
                    print(" " * len(indices_row) + "   " + sleep_art[3])
                    print(" " * len(indices_row) + "   " + sleep_art[4])

                else:
                    pass_art = [
                        "------>",
                        "PASSAR",
                        "[ENTER]"
                    ]

                    print(pieces_row + "   " + pass_art[0])
                    print(indices_row + "   " + pass_art[1])
                    print(" " * len(indices_row) + "   " + pass_art[2])

            else:
                print(pieces_row)
                print(indices_row)

        return playable_tiles
    

    def choice_tile(self, table: Table, playable_tiles: list[Tile]):

        LEFT = {"l", "left", "e", "esquerda"}
        RIGHT = {"r", "right", "d", "direita"}

        tile_idx = int(input("Digite o número equivalente da peça: "))

        if not 0 <= tile_idx < len(playable_tiles):
            raise IndexError("Escolha um índice válido")

        tile = playable_tiles[tile_idx]

        if not table.tiles_deque:
            table.add_first_tile(tile)

        else:
            sides = table.get_sides()

            can_left = (
                tile.left_side == sides[0]
                or tile.right_side == sides[0]
            )
            can_right = (
                tile.left_side == sides[1]
                or tile.right_side == sides[1]
            )

            # escolher automaticamente
            if can_left and not can_right:
                table.add_left(tile)
            elif can_right and not can_left:
                table.add_right(tile)

            elif can_left and can_right:
                side = input(
                    "Digite l (esquerda) ou r (direita): "
                ).strip().lower()

                if side in LEFT:
                    table.add_left(tile)
                elif side in RIGHT:
                    table.add_right(tile)
                else:
                    raise ValueError("Escolha um lado válido")

            else:
                raise ValueError("Essa peça não pode ser jogada")

        self.tiles.remove(tile)


    def pull_tile(self, sleep) -> Tile | None:
        if sleep is not None:
            random.shuffle(sleep)
            return sleep.pop()
        return None


    def play_turn(
            self, 
            table: Table, 
            sleep: list[Tile] | None, 
            pull_one: bool = False
            ) -> None:

        table.show_table()

        while not self.can_play(table) and sleep:

            playable_tiles = self.get_playable_tiles(table, verbose=True, has_sleep=True)
            input("Aperte Enter para comprar uma peça...")

            if pull_one:
                tile = self.pull_tile(sleep)

                if tile is not None:
                    self.tiles.append(tile)

                break

            tile = self.pull_tile(sleep)

            if tile is None:
                break

            self.tiles.append(tile)

        # passar turno
        if not self.can_play(table):
            playable_tiles = self.get_playable_tiles(table, verbose=True)
            input("Você não pode jogar, aperte Enter para passar sua vez")
            print(f"O jogador {self.name} passou a vez")
            return

        playable_tiles = self.get_playable_tiles(table, verbose=True)


        while True:
            try:
                self.choice_tile(table, playable_tiles)
                break
            except (ValueError, IndexError):
                print("Escolha outra peça")
 

In [123]:
class Domino:

    def __init__(self):
        self.players = []
        self.sleep = None

        self.table = Table()

        self.tiles = Domino.make_tiles()
        self.TOTAL_TILES = len(self.tiles)
            


    @staticmethod
    def make_tiles():
        tiles_tup = []
        pivot = 0
        for side1 in range(0, 7):
            for side2 in range(pivot, 7):
                tiles_tup.append((side1, side2))
            pivot+=1
        tiles = [Tile(tup) for tup in tiles_tup]
        return tiles


    @staticmethod
    def get_generic_doubles(tiles_list: list[Tile] | None = None) -> list[Tile]:
        if tiles_list is None:
            tiles_list = Domino.make_tiles()

        return [t for t in tiles_list if t.is_double]


    @staticmethod
    def get_generic_not_doubles(tiles_list: list[Tile] | None = None) -> list[Tile]:
        if tiles_list is None:
            tiles_list = Domino.make_tiles()

        return [t for t in tiles_list if t.is_double]


    @staticmethod
    def get_generic_prioritized_tiles(tiles_list: list[Tile] | None = None) -> list[Tile]:
        if tiles_list is None:
            tiles_list = Domino.make_tiles()

        return (
            Domino.get_generic_doubles(tiles_list)[::-1] +
            sorted(
                Domino.get_generic_not_doubles(tiles_list), 
                key=lambda tile: sum(tile), 
                reverse=True
            )    
        )


    def divide_tiles(
            self,
            n_players: int = 4, 
            tile_count: int = 7,
            has_sleep: bool = False,
        ) -> list[Player]:
        
        if 4 < n_players or n_players < 2:
            raise ValueError("Só é permitido entre 2 a 4 jogadores")
        if tile_count < 5 or len(self.tiles) // n_players < tile_count:
            raise ValueError("Selecione uma quantidade válida de peças")
        
        random.shuffle(self.tiles)

        self.players = [
            Player(
                f"P{n+1}", 
                [self.tiles.pop() for _ in range(tile_count)]
            ) for n in range(n_players)]

        if not has_sleep or len(self.tiles) == 1:
            self.sleep = None
        else:
            self.sleep = self.tiles.copy()
            self.tiles.clear()

        return self.players


    def find_first_player_with_tile(self, tile_list: list[Tile]) -> tuple[Player, Tile]:
        for tile in tile_list:
            for player in self.players:
                if tile in player.tiles:
                    return player, tile
        raise ValueError("Nenhum jogador tem as peças")


    def get_match_podium(self, verbose: bool = False) -> list[Player]:
        podium = sorted(self.players, key=lambda player: player.get_tile_points())

        if verbose:
            print("-"*99)
            print("Placar de pontos da partida")
            print("-"*99)
            pos = 1
            for p in podium:
                print(f"{pos}º - {p.name} - {p.get_tile_points()} points - {p.tiles}")
                pos += 1
            print("-"*99)

        return podium


    def get_game_podium(self, verbose: bool = False) -> list[Player]:
        podium = sorted(self.players, key=lambda player: player.wins, reverse=True)
        
        if verbose:
            print("~=~"*33)
            print("Pódio do jogo")
            print("~=~"*33)
            pos = 1
            for p in podium:
                print(f"{pos}º - {p.name} - {p.wins} wins")
                pos += 1
            print("~=~"*33)

        return podium


    def find_player_with_fewer_points(self) -> Player | None:

        players_ordened_per_points = self.get_match_podium()

        if players_ordened_per_points[0].get_tile_points() == players_ordened_per_points[1].get_tile_points():
            return None

        return players_ordened_per_points[0]
    

    def rotate_players(self, starter: Player) -> None:
        starter_idx = self.players.index(starter)
        self.players = self.players[starter_idx:] + self.players[:starter_idx]


    def verify_starter(self) -> tuple[Player, Tile]:

        prioritized_tiles = self.get_generic_prioritized_tiles()

        try:
            starter, starter_tile = self.find_first_player_with_tile(prioritized_tiles)

        except ValueError:
            starter = random.choice(self.players)
            starter_tile = max(
                starter.tiles, 
                key=lambda tile: tile.left_side + tile.right_side
            )

        self.rotate_players(starter)
        return starter, starter_tile


    def verify_win(self) -> bool:
        for player in self.players:
            if not player.tiles:
                return True
        return False


    def game_is_locked(self, sleep: list[Tile] | None) -> bool:
        if sleep:
            return False
        for player in self.players:
            if player.can_play(self.table):
                return False
        return True


    def has_winner_or_draw(self, player: Player):
        if self.verify_win():
            player.wins+=1
            print(f"\n{player.name} venceu!!!\n")
            return True
        
        if self.game_is_locked(self.sleep):
            winner = self.find_player_with_fewer_points()

            if winner == None:
                print("\nEmpate por pontos.\n")
                return True
            else:
                winner.wins+=1
                print(f"\n{winner.name} venceu por pontos!!!\n")

                self.get_match_podium(verbose=True)
                return True
        return False


    @staticmethod
    def verify_yes_or_not(string: str) -> bool:
        YES = {"y", "yes", "yah", "s", "sim"}
        NOT = {"n", "no", "not", "nao"}

        if string in YES:
            return True
        elif string in NOT:
            return False
        else:
            raise ValueError("Digite y ou n")


    def ask_new_round(self):

        repeat = input("Deseja jogar novamente? (y | n)").strip().lower()
        
        return self.verify_yes_or_not(repeat)



    def define_match_configuration(self):
        
        n_players = int(input("Digite a quantidade de jogadores (2 a 4): "))
        tile_count = int(input("Digite a quantidade de peças para cada jogador (5 a 28/players): "))
        has_sleep_str = input("Terá dorme | puxa | compra? (y | n): ").strip().lower()

        has_sleep = self.verify_yes_or_not(has_sleep_str)

        if has_sleep:
            pull_one_str = input("O dorme será limitado a puxar apenas uma? (y | n): ").strip().lower()
            pull_one = self.verify_yes_or_not(pull_one_str)
            return n_players, tile_count, has_sleep, pull_one

        return n_players, tile_count, has_sleep, None
        

    def reset_game(self):
        self.players = []
        self.sleep = None
        self.table = Table()
        self.tiles = Domino.make_tiles()


    def start_game(self):
        try:
            while True:
                try:
                    n_players, tile_count, has_sleep, pull_one = self.define_match_configuration()
                    break
                except ValueError as error:
                    print(error)

            while True:

                self.divide_tiles(
                    n_players,
                    tile_count,
                    has_sleep,
                )

                self.verify_starter()

                game_ended = False
                while not game_ended:
                    for player in self.players:
                        print(f"\nTurno de {player.name}")
                        player.play_turn(self.table, self.sleep)

                        if self.has_winner_or_draw(player):
                            self.get_game_podium(verbose=True)
                            game_ended = True
                            break

                if not self.ask_new_round():
                    return
                
                self.reset_game()
        except Exception as error:
            print(error)


In [126]:
#TODO 
# - ajeitar classes e pacotes
# - treinar a IA (a pesquisar)
# - fazer a interface gráfica (a decidir)

In [125]:
domino = Domino()

domino.start_game()



Turno de P1
Mesa vazia
 [2|6]   [5|6]   [0|2]   [1|4]   [2|5]  
   0       1       2       3       4    

Turno de P2

---------------------------------------------------------------------------------------------------
[5|6]
---------------------------------------------------------------------------------------------------

 [3|6]   [4|4]   [0|3]   [1|2]   [1|1]  
   0                                    

Turno de P3

---------------------------------------------------------------------------------------------------
[5|6][6|3]
---------------------------------------------------------------------------------------------------

 [2|3]   [2|4]   [0|5]   [0|0]   [0|1]  
   0               1                    

Turno de P1

---------------------------------------------------------------------------------------------------
[0|5][5|6][6|3]
---------------------------------------------------------------------------------------------------

 [2|6]   [0|2]   [1|4]   [2|5]  
           0       